In [42]:
import pandas as pd
import numpy as np
sncf_data = pd.read_csv("project_dataset.csv", sep=";")

np.random.seed(0)
sncf_data.head()



,Date,Service,Gare de dÃ©part,Gare d'arrivÃ©e,DurÃ©e moyenne du trajet,Nombre de circulations prÃ©vues,Nombre de trains annulÃ©s,Commentaire annulations,Nombre de trains en retard au dÃ©part,Retard moyen des trains en retard au dÃ©part,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencÃ©e par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matÃ©riel roulant,Prct retard pour cause gestion en gare et rÃ©utilisation de matÃ©riel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.0,870,5.0,NaN,289.0,11.24780854,...,110.0,6.51,44.0,8.0,36.13445378,31.09243697,10.92436975,15.96638655,"5,04",0.840336134
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.0,406.0,1.0,NaN,213.0,8.479968701,...,32.0,5.363539095,9.0,4.0,20.0,35.0,16.66666667,16.66666667,8.333333333,3.333333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.0,226.0,0.0,NaN,21.0,6.23968254,...,11.0,2.938053097,6.0,1.0,22.22222222,27.77777778,16.66666667,16.66666667,5.555555556,11.11111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,NaN,71.0,7.235211268,...,39.0,5.292211221,18.0,NaN,33.33333333,22.22222222,16.66666667,20.37037037,5.555555556,1.851851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.0,472.0,4.0,NaN,224.0,6.784672619,...,42.0,4.882371795,10.0,0.0,15.78947368,45.61403509,NaN,15.78947368,1.754385965,1.754385965


# Creation Filtre pour DATASET CLEAN 

In [50]:
import re 
#fix noms de colonnes (accents)

def fixaccent(s):
    try:
        return s.encode("latin1").decode("utf-8")
    except Exception:
        return s
sncf_data.columns = [fixaccent(c) for c in sncf_data.columns]
#drop les colonnes commentaire 
comment_cols = [c for c in sncf_data.columns if "Commentaire" in c]
sncf_data = sncf_data.drop(columns=comment_cols, errors="ignore")

#nettoyage colonnes texte 
text_cols = ["Service", "Gare de départ", "Gare d'arrivée"]
for c in text_cols:
    if c in sncf_data.columns:
        sncf_data[c] = sncf_data[c].astype("string").str.strip()

# standardisation gares (pour évite "Paris Montparnasse" vs "PARIS MONTPARNASSE")
for c in ["Gare de départ", "Gare d'arrivée"]:
    if c in sncf_data.columns:
        sncf_data[c] = sncf_data[c].str.upper().str.replace(r"\s+", " ", regex=True)

# 3) Nettoyage Date (formats mixtes: 2018-01, 01-2018, 2018/01, espaces)
def parse_month(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip().str.replace("/", "-", regex=False)
    m = x.str.match(r"^\d{2}-\d{4}$", na=False)
    x.loc[m] = x.loc[m].str.split("-", expand=True).apply(lambda r: f"{r[1]}-{r[0]}", axis=1)
    return pd.to_datetime(x, format="%Y-%m", errors="coerce")

sncf_data["Date"] = parse_month(sncf_data["Date"])
sncf_data["year"] = sncf_data["Date"].dt.year
sncf_data["month"] = sncf_data["Date"].dt.month


# Clean les donnée numérique 

In [ ]:
NA_LIKE = {"NC", "Non communiqué", "NON COMMUNIQUE", "-", ""}

def to_float(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    s = s.replace(list(NA_LIKE), np.nan)

    # enlever "min", gérer virgules, espaces
    s = (s.str.lower()
           .str.replace("min", "", regex=False)
           .str.replace(",", ".", regex=False)
           .str.replace(r"\s+", "", regex=True))

    # extraire le nombre (safe)
    s = s.str.extract(r"(-?\d+(?:\.\d+)?)")[0]
    return pd.to_numeric(s, errors="coerce")

# Colonnes à convertir = tout sauf texte/date
exclude = {"Date", "Service", "Gare de départ", "Gare d'arrivée"}
num_cols = [c for c in sncf_data.columns if c not in exclude and c not in ("year", "month")]

for c in num_cols:
    sncf_data[c] = to_float(sncf_data[c])

# Missing Value

In [ ]:
#Avant de commencer on fait un gros clean des donnes Nul donv NaN  
#On regarde ce qui manque par colonne pour se faire une idée , en % aussi 
missing = sncf_data.isna().sum().sort_values(ascending=False)
total_missing = missing.sum()

report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct_col": sncf_data.isna().mean() * 100,
})

display(report.head(15))
#On remarquera que les commentaire on beaucoup de missing value ce qui est
#  logique , puis en regardant ce que contient les commentaire on se rend 
# compte qu'il y a pas d'interet a garder ces data

,missing_count,missing_pct_col,share_of_all_missing,dtype
Commentaire annulations,11493,95.219553,29.564748,str
Commentaire retards au dÃ©part,11466,94.995857,29.495292,str
Commentaire retards Ã l'arrivÃ©e,10930,90.555095,28.116479,str
Date,60,0.497100,0.154345,str
DurÃ©e moyenne du trajet,240,1.988401,0.617379,str
Gare d'arrivÃ©e,59,0.488815,0.151772,str
Gare de dÃ©part,59,0.488815,0.151772,str
Nombre de circulations prÃ©vues,240,1.988401,0.617379,str
Nombre de trains annulÃ©s,239,1.980116,0.614807,str
Nombre de trains en retard au dÃ©part,240,1.988401,0.617379,str


## Why are they Missing ? Any correlation?


In [ ]:
#avant on a remarqué qu'il y a chaque fois a peu pres 240 valeur nul donc pour êtres sure de ne pas passer a cote de qlq chose qu on pourrai remplacer au lieu de supprimer betement
cols = sncf_data.columns.difference(exclude).tolist()

mask = sncf_data[cols].isna()
# Combien de lignes ont au moins 1 NA dans ces colonnes?
print("Row with any NA:", mask.any(axis=1).sum())
# Combien de lignes ont TOUS ces champs en NA?
print("ROws with ALL NA:", mask.all(axis=1).sum())
# Ya combien de NA par ligne sur ces colones dcp
print(mask.sum(axis=1).value_counts().head(10))

Row with any NA: 4135
ROws with ALL NA: 0
0    7935
1    3379
2     666
3      86
4       4
Name: count, dtype: int64


# Test Logique sncf


In [45]:
#tchek si c'est pas tout simplement liée car de base le nombre de train = 0,(donc tout ce qui peut etres lié a la racine du retard)
mean_cols = [c for c in sncf_data.columns if "Retard moyen" in c]
count_cols = [c for c in sncf_data.columns if "Nombre" in c and "retard" in c.lower()]

print("mean_cols:", len(mean_cols))
print("count_cols:", len(count_cols))

mean_cols: 5
count_cols: 5


# Resultat trop petit , on ne prendra pas en compte ce test

In [ ]:


df = sncf_data.copy()

#  On ignore les colonnes "Commentaire" (quasi vides / inutiles)
comment_cols = [c for c in df.columns if "Commentaire" in c]
df2 = df.drop(columns=comment_cols, errors="ignore")

n_rows = len(df2)

#  Impact d'un dropna global (sur le dataset utile)
after_drop_all = df2.dropna()
pct_kept_all = 100 * len(after_drop_all) / n_rows

# Distribution des NaN par ligne (dataset utile)
na_per_row = df2.isna().sum(axis=1)
max_na = int(na_per_row.max())
pct_rows_any_na = 100 * (na_per_row.gt(0).mean())

# Colonnes les plus touchées (hors commentaires)
missing_summary = (
    pd.DataFrame({
        "missing_count": df2.isna().sum(),
        "missing_pct": df2.isna().mean() * 100
    })
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

print("Missingness report (hors colonnes Commentaire) ")
print(f"Rows total: {n_rows}")
print(f"Rows with any NA: {na_per_row.gt(0).sum()} ({pct_rows_any_na:.1f}%)")
print(f"Max NA per row: {max_na}")
print("\nTop NA-per-row distribution:")
print(na_per_row.value_counts().head(10))

print("\n=== Why we do NOT use dropna() globally ===")
print(f"Rows kept after df.dropna(): {len(after_drop_all)} / {n_rows} ({pct_kept_all:.2f}%)")
print(" Global dropna() would discard too much data, while missing values are sparse (max 4 per row).")
print("\nTop columns with missing values:")
print(missing_summary.head(15))

Missingness report (hors colonnes Commentaire) 
Rows total: 12070
Rows with any NA: 4135 (34.3%)
Max NA per row: 4

Top NA-per-row distribution:
0    7935
1    3379
2     666
3      86
4       4
Name: count, dtype: int64

=== Why we do NOT use dropna() globally ===
Rows kept after df.dropna(): 7935 / 12070 (65.74%)
=> Global dropna() would discard too much data, while missing values are sparse (max 4 per row).

Top columns with missing values:
                                                    missing_count  missing_pct
Prct retard pour cause matÃ©riel roulant                      243     2.013256
Prct retard pour cause infrastructure                         243     2.013256
Retard moyen des trains en retard Ã  l'arrivÃ©e               242     2.004971
Prct retard pour cause gestion en gare et rÃ©ut...            242     2.004971
Prct retard pour causes externes                              242     2.004971
Nombre trains en retard > 15min                               242     2.004971

## Après suppression des colonnes Commentaire, les valeurs manquantes restent éparses (max 4 NaN par ligne) mais touchent ~34% des lignes. Un dropna() global supprimerait une part importante du dataset (≈ un tiers), donc nous privilégions une stratégie de prétraitement + imputation plutôt que la suppression des lignes 

# Filtre Temps 

In [ ]:
col = "DurÃ©e moyenne du trajet"
sncf_data[col]= (
    sncf_data[col].astrype("string").str.lower().str.replace(",", ".", regex:False)
    .str.extract(r"(-?\d+(?:\.\d+)?)")[0].astype(float)
)